# Task 04A — paused A100 Colab study

The standardized-child CPU preflight returned `PAUSE_FULL_STUDY`, so full execution is deliberately disabled. This notebook is retained only to reproduce environment/tests or the bounded preflight. Do not enable the A100 run without a new approved contract. It never pushes to GitHub.


In [ ]:
REPO_URL = 'https://github.com/PaulsonLab/energy-inference-bo.git'
REPO_REF = 'main'
RUN_PREFLIGHT = False
RUN_FULL = False
FULL_STUDY_AUTHORIZED = False  # Repository-controlled scientific gate; do not change manually.


In [ ]:
import os, pathlib, shutil, subprocess, sys
assert (3, 11) <= sys.version_info[:2] <= (3, 12), f'Unsupported Python {sys.version.split()[0]}'
repo = pathlib.Path('/content/energy-inference-bo')
if repo.exists():
    if not (repo/'.git').exists(): raise RuntimeError(f'{repo} exists but is not a Git checkout')
    subprocess.run(['git','fetch','origin','main'],cwd=repo,check=True)
else: subprocess.run(['git','clone','--filter=blob:none',REPO_URL,str(repo)],check=True)
checkout_ref = 'origin/main' if REPO_REF == 'main' else REPO_REF
subprocess.run(['git','checkout','--detach',checkout_ref],cwd=repo,check=True)
sha=subprocess.check_output(['git','rev-parse','HEAD'],cwd=repo,text=True).strip(); print('Git SHA:',sha)
if shutil.which('nvidia-smi'): subprocess.run(['nvidia-smi'],check=True)
os.environ['MPLBACKEND']='Agg'
subprocess.run([sys.executable,'-m','pip','install','--no-cache-dir','-q','uv==0.10.11'],check=True)
uv_cmd=[sys.executable,'-m','uv']; study_venv=repo/'.venv'
if study_venv.exists(): shutil.rmtree(study_venv)
subprocess.run([*uv_cmd,'sync','--locked','--group','dev'],cwd=repo,env=os.environ.copy(),check=True)
study_python=study_venv/'bin/python'; scientific_python=[str(study_python)]
scientific_env=os.environ.copy(); scientific_env['MPLBACKEND']='Agg'


In [ ]:
import json
probe="""import json,sys,torch; name=torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'; print(json.dumps({'python':sys.version,'torch':torch.__version__,'cuda':torch.version.cuda,'cuda_available':torch.cuda.is_available(),'device':name}))"""
result=subprocess.run([*scientific_python,'-c',probe],text=True,capture_output=True,env=scientific_env)
if result.returncode: raise RuntimeError(result.stderr)
runtime=json.loads(result.stdout); print(runtime)
if FULL_STUDY_AUTHORIZED: assert runtime['cuda_available'] and 'A100' in runtime['device'], f"Task 04A full profile requires A100, got {runtime['device']}"
subprocess.run([*scientific_python,'-m','pytest','-q'],cwd=repo,env=scientific_env,check=True)


In [ ]:
if RUN_PREFLIGHT:
    subprocess.run([*scientific_python,'-m','energy_bo.experiments.run_task04a','--profile','preflight','--device','cpu','--output-dir','artifacts/task04a/colab_preflight'],cwd=repo,env=scientific_env,check=True)
else: print('Preflight reproduction skipped. The reviewed local result is PAUSE_FULL_STUDY.')


In [ ]:
if RUN_FULL:
    if not FULL_STUDY_AUTHORIZED: raise RuntimeError('Full Task 04A is paused by its frozen CPU preflight. A new approved contract is required.')
    preflight=json.loads((repo/'artifacts/task04a/colab_preflight/gate_status.json').read_text())
    if preflight['decision'] != 'READY_FOR_FULL': raise RuntimeError(f'Preflight did not authorize full execution: {preflight}')
    command=[*scientific_python,'-m','energy_bo.experiments.run_task04a','--profile','full','--device','cuda','--output-dir','artifacts/task04a/full']
    subprocess.run(command,cwd=repo,env=scientific_env,check=True)
    out=repo/'artifacts/task04a/full'; manifest={'git_sha':sha,'runtime':runtime,'command':command}
    (out/'colab_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
    archive=shutil.make_archive('/content/task04a_full_outputs','zip',root_dir=out)
    from google.colab import files; files.download(archive)
else: print('Full study is intentionally disabled by the Task 04A preflight.')


## Status

There should be no full ZIP under the current contract. If a future approved revision re-enables the study, retain raw output under `artifacts/task04a/full/` and import only audited compact evidence into `results/`.
